## Setup

In [1]:
from statsbombpy import sb
import matplotlib.pyplot as plt
import numpy as np
import requests
import json
import pandas as pd

In [2]:
# position id to on-pitch position mapping
# Pitch dimensions are 120 x 80 (meters). (0,0) is at top left corner. x increases to the right, y increases downwards.
dimensions = [120, 80]
def position_id_to_coordinates(position_id):
    mapping = {
        1: (8, 40),    # GK
        2: (20, 75),   # RB
        3: (20, 55),   # RCB
        4: (20, 40),   # CB
        5: (20, 25),   # LCB
        6: (20, 5),    # LB
        7: (35, 75),   # RWB
        8: (35, 5),    # LWB
        9: (45, 55),   # RDM
        10: (45, 40),  # CDM
        11: (45, 25),  # LDM
        12: (60, 75),  # RM
        13: (60, 55),  # RCM
        14: (60, 40),  # CM (center circle)
        15: (60, 25),  # LCM
        16: (60, 5),  # LM
        17: (90, 75),  # RW
        18: (80, 55),  # RAM
        19: (80, 40),  # CAM
        20: (80, 25),  # LAM
        21: (90, 5),  # LW
        22: (105, 55), # RCF
        23: (110, 40), # ST
        24: (105, 25), # LCF
        25: (98, 40),  # SS (second striker)
    }
    return mapping.get(position_id, None)

## Map the 11

In [61]:
competition_id = 9
season_id = 281
match_id = 3895210
matches = sb.matches(competition_id=competition_id, season_id=season_id)
match_info = matches[matches["match_id"] == match_id]
events = sb.events(match_id=match_id)
team_names = list(sb.lineups(match_id).keys())
score = match_info[["home_score", "away_score"]].iloc[0] if not match_info.empty else {"home_score": "Unknown", "away_score": "Unknown"}
#formations =  events[events["type"] == "Starting XI"]["tactics"].apply(lambda t: t.get("formation") if isinstance(t, dict) else None)
date_of_match = match_info["match_date"].iloc[0] if not match_info.empty else "Unknown"
competition = match_info["competition"].iloc[0] if not match_info.empty else "Unknown"
round = match_info["match_week"].iloc[0] if not match_info.empty else "Unknown"
print(f"Date: {date_of_match}, Competition: {competition}, Round: {round}")
print(f"{team_names[0]} {score['home_score']} - {score['away_score']} {team_names[1]}")
print(f"Starting XI")

def get_team_lineup(team):
	formation = (
		events.loc[(events["team"] == team) & (events["type"] == "Starting XI"), "tactics"]
		.apply(lambda t: t.get("formation") if isinstance(t, dict) else None)
		.dropna()
	)
	formation = formation.iloc[0] if not formation.empty else None
	# turn the format of formation from an integer "433" to string "4-3-3" or "4231" to "4-2-3-1"
	if formation is not None:
		formation_str = str(formation)
		formation = "-".join(formation_str)
	team_lineup = sb.lineups(match_id=match_id)[team]
	lineup = team_lineup.assign(
		start_reason=team_lineup["positions"].apply(
			lambda p: p[0].get("start_reason") if isinstance(p, list) and len(p) > 0 else None
		),
		position=team_lineup["positions"].apply(
			lambda p: p[0].get("position") if isinstance(p, list) and len(p) > 0 else None
		),
		position_id=team_lineup["positions"].apply(
			lambda p: p[0].get("position_id") if isinstance(p, list) and len(p) > 0 else None
		),
	)[["player_name", "player_nickname", "jersey_number", "start_reason", "position", "position_id"]]
	lineup["player_nickname"] = lineup["player_nickname"].fillna(team_lineup["player_name"])
	lineup = lineup.drop(columns=["player_name"], errors="ignore").rename(columns={"player_nickname": "player_name"})
	lineup = lineup[lineup["start_reason"] == "Starting XI"]
	lineup = lineup.drop(columns=["start_reason"], errors="ignore")
	return formation, lineup

for team in team_names:
	formation, team_lineup = get_team_lineup(team)
	print(f"Team: {team}, Formation: {formation}")
	print(team_lineup[["player_name", "position"]])
	print("\n")

/home/chrischu/miniconda3/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/home/chrischu/miniconda3/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Date: 2024-01-27, Competition: Germany - 1. Bundesliga, Round: 19
Bayer Leverkusen 0 - 0 Borussia Mönchengladbach
Starting XI
Team: Bayer Leverkusen, Formation: 3-4-3
         player_name                  position
1       Granit Xhaka  Right Defensive Midfield
2      Patrik Schick            Center Forward
3       Nadiem Amiri   Left Defensive Midfield
5      Jonas Hofmann                Right Wing
6     Robert Andrich               Center Back
7      Álex Grimaldo            Left Wing Back
9   Jeremie Frimpong           Right Wing Back
10    Piero Hincapié          Left Center Back
11     Florian Wirtz                 Left Wing
13       Matěj Kovář                Goalkeeper
14    Josip Stanišić         Right Center Back


Team: Borussia Mönchengladbach, Formation: 3-5-2
          player_name                   position
0      Franck Honorat            Right Wing Back
1   Jordan Siebatcheu       Right Center Forward
4         Nico Elvedi           Left Center Back
5        Julian Weigl 

/home/chrischu/miniconda3/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/home/chrischu/miniconda3/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/home/chrischu/miniconda3/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [86]:
# plot the formations of both teams using matplotlib
formation1, lineup1 = get_team_lineup(team_names[0])
formation2, lineup2 = get_team_lineup(team_names[1])

plt.figure(figsize=(13, 8))
for i, team in enumerate(team_names):
    plt.subplot(1, 2, i+1)
    plt.title(f"{team} - Formation: {formation1 if i == 0 else formation2}")
    plt.xlim(0, 80)
    plt.ylim(0, 120)
    plt.xticks([])
    plt.yticks([])
    plt.gca().set_facecolor('lightgreen')

    for _, player in (lineup1 if i == 0 else lineup2).iterrows():
        pos_coords = position_id_to_coordinates(player["position_id"])
        player_first_name = player["player_name"].split()[0] if isinstance(player["player_name"], str) else "Unknown"
        player_last_name = player["player_name"].split()[-1] if isinstance(player["player_name"], str) else "Unknown"
        if pos_coords is not None:
            plt.scatter(pos_coords[1], pos_coords[0], s=400, color='blue', edgecolors='black', zorder=5)
            plt.text(pos_coords[1], pos_coords[0], player["jersey_number"], ha='center', va='center', fontsize=9, fontweight='bold', color='white', zorder=6)
            plt.text(pos_coords[1], pos_coords[0] - 4, f"{player_first_name}", ha='center', va='center', fontsize=11, zorder=6)
            plt.text(pos_coords[1], pos_coords[0] - 7, f"{player_last_name}", ha='center', va='center', fontsize=11, zorder=6)
plt.tight_layout()
plt.savefig('/home/chrischu/sports_analytics/formations.png', dpi=300)
plt.show()

/home/chrischu/miniconda3/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/home/chrischu/miniconda3/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/tmp/ipykernel_1120989/166591291.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Find the match id

In [ ]:
# User input league, season and match to visualize the formations of both teams. For example, you can input "Premier League", "2022/2023", and a specific match id to visualize the formations of the teams in that match.
# Example input: "Bundesliga", "2022/2023", "Leverkusen", "Bayern Munich"
# Give out the match id based on the user input of league, season and teams. 



## Other

In [34]:
sb.competitions()

/home/chrischu/miniconda3/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,2025-07-06T04:26:07.636270,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,None,None,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2024-09-28T01:57:35.846538,None,None,2024-09-28T01:57:35.846538
3,16,4,Europe,Champions League,male,False,False,2018/2019,2025-05-08T15:10:50.835274,2021-06-13T16:17:31.694,None,2025-05-08T15:10:50.835274
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,None,2024-02-13T02:35:28.134882
...,...,...,...,...,...,...,...,...,...,...,...,...
70,35,75,Europe,UEFA Europa League,male,False,False,1988/1989,2024-02-12T14:45:05.702250,2021-06-13T16:17:31.694,None,2024-02-12T14:45:05.702250
71,53,315,Europe,UEFA Women's Euro,female,False,True,2025,2025-07-28T14:19:20.467348,2025-07-29T16:03:07.355174,2025-07-29T16:03:07.355174,2025-07-28T14:19:20.467348
72,53,106,Europe,UEFA Women's Euro,female,False,True,2022,2024-02-13T13:27:17.178263,2024-02-13T13:30:52.820588,2024-02-13T13:30:52.820588,2024-02-13T13:27:17.178263
73,72,107,International,Women's World Cup,female,False,True,2023,2025-07-14T10:07:06.620906,2025-07-14T10:10:27.224586,2025-07-14T10:10:27.224586,2025-07-14T10:07:06.620906


In [3]:
# print all competitions with competition id, season id, name, and season
competitions = sb.competitions()
for index, row in competitions.iterrows():
    print(f"ID: {row['competition_id']}, Season ID: {row['season_id']}, Name: {row['competition_name']}, Season: {row['season_name']}")

/home/chrischu/miniconda3/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


ID: 9, Season ID: 281, Name: 1. Bundesliga, Season: 2023/2024
ID: 9, Season ID: 27, Name: 1. Bundesliga, Season: 2015/2016
ID: 1267, Season ID: 107, Name: African Cup of Nations, Season: 2023
ID: 16, Season ID: 4, Name: Champions League, Season: 2018/2019
ID: 16, Season ID: 1, Name: Champions League, Season: 2017/2018
ID: 16, Season ID: 2, Name: Champions League, Season: 2016/2017
ID: 16, Season ID: 27, Name: Champions League, Season: 2015/2016
ID: 16, Season ID: 26, Name: Champions League, Season: 2014/2015
ID: 16, Season ID: 25, Name: Champions League, Season: 2013/2014
ID: 16, Season ID: 24, Name: Champions League, Season: 2012/2013
ID: 16, Season ID: 23, Name: Champions League, Season: 2011/2012
ID: 16, Season ID: 22, Name: Champions League, Season: 2010/2011
ID: 16, Season ID: 21, Name: Champions League, Season: 2009/2010
ID: 16, Season ID: 41, Name: Champions League, Season: 2008/2009
ID: 16, Season ID: 39, Name: Champions League, Season: 2006/2007
ID: 16, Season ID: 37, Name: Ch

In [47]:
sb.matches(competition_id=9, season_id=281)

,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,match_status,...,last_updated_360,match_week,competition_stage,stadium,referee,home_managers,away_managers,data_version,shot_fidelity_version,xy_fidelity_version
0,3895302,2024-04-14,17:30:00.000,Germany - 1. Bundesliga,2023/2024,Bayer Leverkusen,Werder Bremen,5,0,available,...,2024-05-10T17:03:59.613154,29,Regular Season,BayArena,Harm Osmers,Xabier Alonso Olano,Ole Werner,1.1.0,2,2
1,3895292,2024-04-06,15:30:00.000,Germany - 1. Bundesliga,2023/2024,Union Berlin,Bayer Leverkusen,0,1,available,...,2024-05-12T21:08:37.897296,28,Regular Season,Stadion An der Alten Försterei,Benjamin Brand,Nenad Bjelica,Xabier Alonso Olano,1.1.0,2,2
2,3895333,2024-05-05,18:30:00.000,Germany - 1. Bundesliga,2023/2024,Eintracht Frankfurt,Bayer Leverkusen,1,5,available,...,2024-05-14T16:32:13.483516,32,Regular Season,Deutsche Bank Park,Christian Dingert,Dino Toppmöller,Xabier Alonso Olano,1.1.0,2,2
3,3895340,2024-05-12,20:30:00.000,Germany - 1. Bundesliga,2023/2024,Bochum,Bayer Leverkusen,0,5,available,...,2024-05-14T16:46:08.459843,33,Regular Season,Vonovia Ruhrstadion,Benjamin Brand,Heiko Butscher,Xabier Alonso Olano,1.1.0,2,2
4,3895348,2024-05-18,16:30:00.000,Germany - 1. Bundesliga,2023/2024,Bayer Leverkusen,Augsburg,2,1,available,...,2024-05-20T10:33:09.140760,34,Regular Season,BayArena,Matthias Jöllenbeck,Xabier Alonso Olano,Jess Christian Thorup,1.1.0,2,2
5,3895286,2024-03-30,16:30:00.000,Germany - 1. Bundesliga,2023/2024,Bayer Leverkusen,Hoffenheim,2,1,available,...,2024-05-08T03:36:21.320065,27,Regular Season,BayArena,Deniz Aytekin,Xabier Alonso Olano,Pellegrino Matarazzo,1.1.0,2,2
6,3895220,2024-02-03,16:30:00.000,Germany - 1. Bundesliga,2023/2024,Darmstadt 98,Bayer Leverkusen,0,2,available,...,2024-05-08T01:01:15.978489,20,Regular Season,Merck-Stadion am Böllenfalltor,Tobias Reichel,Torsten Lieberknecht,Xabier Alonso Olano,1.1.0,2,2
7,3895250,2024-02-23,21:30:00.000,Germany - 1. Bundesliga,2023/2024,Bayer Leverkusen,FSV Mainz 05,2,1,available,...,2024-05-08T00:38:33.864825,23,Regular Season,BayArena,Timo Gerach,Xabier Alonso Olano,Bo Henriksen,1.1.0,2,2
8,3895266,2024-03-10,20:30:00.000,Germany - 1. Bundesliga,2023/2024,Bayer Leverkusen,Wolfsburg,2,0,available,...,2024-04-28T10:29:53.455142,25,Regular Season,BayArena,Daniel Siebert,Xabier Alonso Olano,Niko Kovač,1.1.0,2,2
9,3895275,2024-03-17,16:30:00.000,Germany - 1. Bundesliga,2023/2024,Freiburg,Bayer Leverkusen,2,3,available,...,2024-04-19T22:39:18.935666,26,Regular Season,Europa-Park Stadion,Harm Osmers,Christian Streich,Xabier Alonso Olano,1.1.0,2,2


In [48]:
sb.lineups(match_id=3895210)["Borussia Mönchengladbach"]

/home/chrischu/miniconda3/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,player_id,player_name,player_nickname,jersey_number,country,cards,positions
0,7198,Franck Honorat,None,9,France,[],"[{'position_id': 7, 'position': 'Right Wing Ba..."
1,7375,Theoson Jordan Siebatcheu,Jordan Siebatcheu,13,United States of America,[],"[{'position_id': 22, 'position': 'Right Center..."
2,8779,Stefan Lainer,None,18,Austria,[],"[{'position_id': 7, 'position': 'Right Wing Ba..."
3,8805,Christoph Kramer,None,23,Germany,[],"[{'position_id': 15, 'position': 'Left Center ..."
4,8814,Nico Elvedi,None,30,Switzerland,[],"[{'position_id': 5, 'position': 'Left Center B..."
5,8816,Julian Weigl,None,8,Germany,"[{'time': '80:47', 'card_type': 'Yellow Card',...","[{'position_id': 10, 'position': 'Center Defen..."
6,9174,Marvin Friedrich,None,5,Germany,[],"[{'position_id': 4, 'position': 'Center Back',..."
7,11399,Robin Hack,None,25,Germany,[],"[{'position_id': 24, 'position': 'Left Center ..."
8,12320,Tony Jantschke,None,24,Germany,[],[]
9,12323,Florian Neuhaus,None,10,Germany,[],"[{'position_id': 15, 'position': 'Left Center ..."


In [77]:
# print the position for each player in the starting lineup
lineups = sb.lineups(match_id=3895210)
for _, player in lineups["Borussia Mönchengladbach"].iterrows():
    print(f"Player: {player['player_name']}, Position: {player['positions']}")

Player: Franck Honorat, Position: [{'position_id': 7, 'position': 'Right Wing Back', 'from': '00:00', 'to': '69:35', 'from_period': 1, 'to_period': 2, 'start_reason': 'Starting XI', 'end_reason': 'Substitution - Off (Tactical)'}]
Player: Theoson Jordan Siebatcheu, Position: [{'position_id': 22, 'position': 'Right Center Forward', 'from': '00:00', 'to': '78:59', 'from_period': 1, 'to_period': 2, 'start_reason': 'Starting XI', 'end_reason': 'Substitution - Off (Tactical)'}]
Player: Stefan Lainer, Position: [{'position_id': 7, 'position': 'Right Wing Back', 'from': '69:35', 'to': None, 'from_period': 2, 'to_period': None, 'start_reason': 'Substitution - On (Tactical)', 'end_reason': 'Final Whistle'}]
Player: Christoph Kramer, Position: [{'position_id': 15, 'position': 'Left Center Midfield', 'from': '78:59', 'to': None, 'from_period': 2, 'to_period': None, 'start_reason': 'Substitution - On (Tactical)', 'end_reason': 'Final Whistle'}]
Player: Nico Elvedi, Position: [{'position_id': 5, 'po

In [50]:
sb.events(match_id=3895210)

,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_recovery_failure,block_deflection,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,...,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 343, 'lineup': [{'player': {'id'...",Bayer Leverkusen,904,00:00:00.000,Starting XI,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 352, 'lineup': [{'player': {'id'...",Borussia Mönchengladbach,185,00:00:00.000,Starting XI,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:00:00.000,Half Start,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Borussia Mönchengladbach,185,00:00:00.000,Half Start,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:00:00.000,Half Start,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 352, 'lineup': [{'player': {'id'...",Borussia Mönchengladbach,185,00:16:35.443,Tactical Shift,NaN
4889,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 343, 'lineup': [{'player': {'id'...",Bayer Leverkusen,904,00:22:08.052,Tactical Shift,NaN
4890,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:26:03.089,Shield,True
4891,NaN,Yellow Card,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Bayer Leverkusen,904,00:28:45.711,Bad Behaviour,NaN


In [109]:
match_id = 3895210
events = sb.events(match_id=match_id)
starting = events[events["type"] == "Starting XI"].copy()
starting["formation"] = starting["tactics"].apply(lambda t: t.get("formation") if isinstance(t, dict) else None)
print(starting[["team", "formation"]])

                       team  formation
0          Bayer Leverkusen        343
1  Borussia Mönchengladbach        352


/home/chrischu/miniconda3/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [43]:
# open data straight from the github repo

match_id = 3895210
link_360 = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/three-sixty/{match_id}.json"
grab = requests.get(link_360)
new_data = grab.json()

# use pandas to convert the data into a dataframe
pd.json_normalize(new_data)

,event_uuid,visible_area,freeze_frame
0,130ef057-a5f4-423d-8529-1fc7f92d01b8,"[120.0, 80.0, 0.0, 80.0, 0.0, 58.9367036862396...","[{'teammate': True, 'actor': False, 'keeper': ..."
1,99cdc341-1779-4e3a-a716-0bf10400ce5d,"[0.0, 80.0, 0.0, 78.9486324092321, 36.52523863...","[{'teammate': True, 'actor': True, 'keeper': F..."
2,bf2d3490-fd6e-42b8-9440-ae43493928dd,"[0.0, 80.0, 0.0, 78.9486324092321, 36.52523863...","[{'teammate': True, 'actor': True, 'keeper': F..."
3,12cb81da-9d37-4d66-b139-5d9ff0064af2,"[5.007939676118557, 80.0, 37.538331070765665, ...","[{'teammate': True, 'actor': True, 'keeper': F..."
4,a72cc5e2-e26f-4aca-8dde-37500741049c,"[22.533036018283404, 80.0, 0.0, 52.35780736126...","[{'teammate': True, 'actor': False, 'keeper': ..."
...,...,...,...
4384,8858dffb-8115-49c9-a3ad-907607bd3b99,"[105.31194194903485, 75.69853477878341, 81.627...","[{'teammate': False, 'actor': False, 'keeper':..."
4385,c4b5b41e-4ad6-40be-ac4e-4f212c268056,"[0.0, 30.711402691707452, 15.034261323393423, ...","[{'teammate': True, 'actor': False, 'keeper': ..."
4386,5c3e70ac-3117-4546-a0a2-094887fef9bf,"[0.0, 80.0, 0.0, 26.15132382686525, 15.9063707...","[{'teammate': True, 'actor': False, 'keeper': ..."
4387,ba29829b-5761-4de4-9573-070ac27b41b1,"[102.931414601768, 74.79021394409415, 79.69063...","[{'teammate': True, 'actor': False, 'keeper': ..."


In [22]:
# print all possible event types
events = sb.events(match_id=3895210)
print(events["type"].unique())

['Starting XI' 'Half Start' 'Pass' 'Ball Receipt*' 'Carry' 'Clearance'
 'Pressure' 'Interception' 'Ball Recovery' 'Dispossessed' 'Duel'
 'Foul Committed' 'Foul Won' 'Shot' 'Goal Keeper' 'Block' 'Miscontrol'
 '50/50' 'Dribbled Past' 'Dribble' 'Error' 'Half End' 'Injury Stoppage'
 'Referee Ball-Drop' 'Offside' 'Substitution' 'Tactical Shift' 'Shield'
 'Bad Behaviour']
